In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 60 \
    --horizon 10 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [1]:
!python run_walkforward.py \
    --dataset_dir "data/processed/2000_2026_1d_60_10" \
    --runs 100 \
    --batch_size 8192 \
    --epochs 100 \
    --l2_reg "1e-4" \
    --lr "1e-3" \
    --start_fold "fold_2020" \
    --append

🚀 Запуск массового обучения моделей (Walk-Forward)...
📁 Датасет: data/processed/2000_2026_1d_60_10
⚙️  Настройки: 100 runs, 100 epochs, batch 8192
⏭️ Пропускаем завершенные фолды. Начинаем строго с: fold_2020

🔥 Обучение нейросети для: fold_2020
✅ Mixed precision включена!
✅ Динамическое выделение видеопамяти включено!
➕ Флаг --append: Модели в фолде [fold_2020] существуют. Обучаем новые поверх старых.
🚀 Старт обучения. Фолд: [fold_2020]
📊 Форма данных: [Lookback: 60, Features: 70]
⚙️ Расчет идеальных весов классов...
   Баланс: SL(0)=32528, Hold(1)=79222, TP(2)=32183
   Веса:   SL(0)=1.47, Hold(1)=0.61, TP(2)=1.49
⏳ Подготовка конвейера данных...

--------------------------------------------------
🔄 ИТЕРАЦИЯ 1/100 (Лучшая точность сессии: 0.00%)
--------------------------------------------------
Epoch 1/100
18/18 - 12s - 656ms/step - accuracy: 0.4509 - loss: 1.2373 - val_accuracy: 0.5073 - val_loss: 1.1683
Epoch 2/100
18/18 - 7s - 412ms/step - accuracy: 0.4999 - loss: 1.0967 - val_acc

In [ ]:
#очистка наименне успешных ltsm моделей (остается топ 3)
!python -m _tools.clean_lstm_models

In [ ]:
#формирование последовательностей предсказаний
!python -m _tools.generate_predictions

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.train_rllib_pbt --population 4 --iterations 3000 --force